# Các file cần thiết: 
 1. LST CELL.xlsx
 2. LST EUCELLSECTOREQM.xlsx
 3. LST SECTOREQM.xlsx
 4. Inventory_Board.csv
 5. LST CELLOP.xlsx
 6. LST CNOPERATORTA.xlsx

In [ ]:
import pandas as pd

In [ ]:
def read_excel_columns(file_path, columns, header_row=1, rename_dict=None):
    """
    Đọc file Excel, lấy các cột cần thiết, đổi tên cột nếu cần.
    file_path: đường dẫn file Excel
    columns: danh sách tên cột cần lấy
    header_row: số thứ tự dòng header (mặc định 1, tức là dòng thứ 2)
    rename_dict: dict đổi tên cột nếu cần (mặc định None)
    """
    df = pd.read_excel(file_path, header=header_row, engine='openpyxl')
    df = df[[col for col in columns if col in df.columns]]
    if rename_dict:
        df = df.rename(columns=rename_dict)
    return df

# Ví dụ sử dụng:
# df_cell = read_excel_columns('LST CELL_2025_05_27_11_34_17_832.xlsx', columns_needed)
# df_cell_sectoreq = read_excel_columns('LST EUCELLSECTOREQM_2025_05_27_11_35_12_418.xlsx', columns_eucell_needed, rename_dict={'Sector equipment ID': 'Sector Equipment ID'})

In [ ]:
# Định nghĩa các cột cần lấy cho từng file
columns_cell = [
    'Base Station Name', 'Local Cell ID', 'Cell Name', 'Frequency band',
    'Downlink EARFCN', 'Downlink bandwidth', 'Cell transmission and reception mode', 'Physical cell ID' , 'Root sequence index', 'Cell radius(m)'
]
columns_eucell = [
    'Base Station Name', 'Local cell ID', 'Sector equipment ID'
]
columns_sectoreq = [
    'Base Station Name', 'Sector Equipment ID', 'Sector Equipment Antenna:Subrack No.'
]

# Đọc dữ liệu từ các file
file_cell = 'LST CELL.xlsx'
file_eucell = 'LST EUCELLSECTOREQM.xlsx'
file_sectoreq = 'LST SECTOREQM.xlsx'

df_cell = read_excel_columns(file_cell, columns_cell)
df_cell_sectoreq = read_excel_columns(
    file_eucell, columns_eucell, rename_dict={'Sector equipment ID': 'Sector Equipment ID'}
)
df_sectoreq = read_excel_columns(file_sectoreq, columns_sectoreq)
df_sectoreq = df_sectoreq.drop_duplicates()

# Gộp các bảng theo yêu cầu
# Gộp df_cell và df_cell_sectoreq
merged1 = pd.merge(
    df_cell,
    df_cell_sectoreq,
    how='inner',
    left_on=['Base Station Name', 'Local Cell ID'],
    right_on=['Base Station Name', 'Local cell ID']
)
merged1 = merged1.drop(columns=['Local cell ID'])

# Gộp tiếp với df_sectoreq
final_df = pd.merge(
    merged1,
    df_sectoreq,
    how='inner',
    on=['Base Station Name', 'Sector Equipment ID']
)

final_df.head()

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Base Station Name,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Sector Equipment Antenna:Subrack No.
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0


In [ ]:
# Đọc file Inventory_Board_20250527_111941.csv cho BBU và RRU
file_board = 'Inventory_Board.csv'

# Các cột cho BBU
columns_bbu = ['NEName', 'Board Name', 'Board Type']
df_bbu = pd.read_csv(file_board, usecols=[col for col in columns_bbu if col])
# Lọc các dòng có chữ 'UBBP' trong cột Board Name
mask_ubbp = df_bbu['Board Name'].astype(str).str.contains('UBBP', na=False)
df_bbu = df_bbu[mask_ubbp]
# Dùng regex lấy dữ liệu bắt đầu bằng 'UBB' cho đến hết dòng trong Board Type
import re
def extract_ubb(text):
    match = re.search(r'(UBB\w*)', str(text))
    return match.group(1) if match else ''
df_bbu['Board Type'] = df_bbu['Board Type'].apply(extract_ubb)

# Các cột cho RRU (không lấy Board Type)
columns_rru = ['NEName', 'Board Name', 'Subrack No.', 'Manufacturer Data']
df_rru = pd.read_csv(file_board, usecols=[col for col in columns_rru if col])
# Lọc các dòng có chữ 'MRRU' trong cột Board Name
mask_mrru = df_rru['Board Name'].astype(str).str.contains('MRRU', na=False)
df_rru = df_rru[mask_mrru]
# Giữ lại thông tin trước dấu phẩy đầu tiên trong Manufacturer Data
if 'Manufacturer Data' in df_rru.columns:
    df_rru['Manufacturer Data'] = df_rru['Manufacturer Data'].astype(str).str.split(',').str[0]
    df_rru = df_rru.rename(columns={'Manufacturer Data': 'RF Type'})

df_bbu.head(), df_rru.head()

(    NEName Board Name Board Type
 2   AGTS33       UBBP     UBBPe2
 14  AGTB17       UBBP     UBBPd4
 28  AGTB20       UBBP     UBBPe2
 53  CMDD30       UBBP     UBBPd4
 55  CMDD35       UBBP     UBBPe2,
     NEName Board Name  Subrack No.   RF Type
 9   AGTS33       MRRU           62  RRU3971a
 10  AGTS33       MRRU           61  RRU3971a
 11  AGTS33       MRRU           60  RRU3971a
 18  AGTB17       MRRU           60  RRU3971a
 19  AGTB17       MRRU           61  RRU3971a)

In [ ]:
df_bbu.head(10)

,NEName,Board Name,Board Type
2,AGTS33,UBBP,UBBPe2
14,AGTB17,UBBP,UBBPd4
28,AGTB20,UBBP,UBBPe2
53,CMDD30,UBBP,UBBPd4
55,CMDD35,UBBP,UBBPe2
83,CMDD42,UBBP,UBBPe2
95,CMTT34,UBBP,UBBPd4
111,CMNC23,UBBP,UBBPe2
122,STVC39,UBBP,UBBPe2
135,STMX21,UBBP,UBBPd4


In [ ]:
df_bbu.info()
df_bbu.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 8029 entries, 2 to 99383
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   NEName      8029 non-null   object
 1   Board Name  8029 non-null   object
 2   Board Type  8029 non-null   object
dtypes: object(3)
memory usage: 250.9+ KB


,NEName,Board Name,Board Type
count,8029,8029,8029
unique,5765,1,10
top,CTOM37,UBBP,UBBPe2
freq,4,8029,1880


In [ ]:

# Thêm cột đếm số lượng cho mỗi cặp NEName và Board Type
df_bbu['Count'] = df_bbu.groupby(['NEName','Board Type'])['Board Type'].transform('count')     
df_bbu.head(10)

,NEName,Board Name,Board Type,Count
2,AGTS33,UBBP,UBBPe2,1
14,AGTB17,UBBP,UBBPd4,1
28,AGTB20,UBBP,UBBPe2,1
53,CMDD30,UBBP,UBBPd4,1
55,CMDD35,UBBP,UBBPe2,1
83,CMDD42,UBBP,UBBPe2,1
95,CMTT34,UBBP,UBBPd4,1
111,CMNC23,UBBP,UBBPe2,1
122,STVC39,UBBP,UBBPe2,1
135,STMX21,UBBP,UBBPd4,1


In [ ]:
df_bbu.sort_values(by=['NEName', 'Count'], inplace=True)
df_bbu.head(10)

,NEName,Board Name,Board Type,Count
66657,AGAP01,UBBP,UBBPd4,1
49336,AGAP02,UBBP,UBBPe2,1
49616,AGAP03,UBBP,UBBPe2,1
969,AGAP04,UBBP,UBBPe2,1
66669,AGAP05,UBBP,UBBPd4,1
49315,AGAP06,UBBP,UBBPe2,1
29591,AGAP07,UBBP,UBBPe2,1
32316,AGAP08,UBBP,UBBPd4,1
32318,AGAP08,UBBP,UBBPe2,1
90106,AGAP08NR77,UBBP,UBBPg3a,1


In [ ]:
df_bbu['Config'] = df_bbu['Count'].astype(str) + '_' + df_bbu['Board Type'].astype(str)
df_bbu.head()

,NEName,Board Name,Board Type,Count,Config
66657,AGAP01,UBBP,UBBPd4,1,1_UBBPd4
49336,AGAP02,UBBP,UBBPe2,1,1_UBBPe2
49616,AGAP03,UBBP,UBBPe2,1,1_UBBPe2
969,AGAP04,UBBP,UBBPe2,1,1_UBBPe2
66669,AGAP05,UBBP,UBBPd4,1,1_UBBPd4


In [ ]:
df_bbu = df_bbu[['NEName','Config']].reset_index(drop=True)
df_bbu = df_bbu.drop_duplicates()
df_bbu.head(10)

,NEName,Config
0,AGAP01,1_UBBPd4
1,AGAP02,1_UBBPe2
2,AGAP03,1_UBBPe2
3,AGAP04,1_UBBPe2
4,AGAP05,1_UBBPd4
5,AGAP06,1_UBBPe2
6,AGAP07,1_UBBPe2
7,AGAP08,1_UBBPd4
8,AGAP08,1_UBBPe2
9,AGAP08NR77,1_UBBPg3a


In [ ]:
#df["Danh sách điểm"] = (df.groupby("Họ tên")["Điểm"].transform(lambda x: ", ".join(map(str, x))))
df_bbu['Config_BBU'] = (df_bbu.groupby("NEName")["Config"].transform(lambda x: "+".join(map(str, x))))
df_bbu.head(10)

,NEName,Config,Config_BBU
0,AGAP01,1_UBBPd4,1_UBBPd4
1,AGAP02,1_UBBPe2,1_UBBPe2
2,AGAP03,1_UBBPe2,1_UBBPe2
3,AGAP04,1_UBBPe2,1_UBBPe2
4,AGAP05,1_UBBPd4,1_UBBPd4
5,AGAP06,1_UBBPe2,1_UBBPe2
6,AGAP07,1_UBBPe2,1_UBBPe2
7,AGAP08,1_UBBPd4,1_UBBPd4+1_UBBPe2
8,AGAP08,1_UBBPe2,1_UBBPd4+1_UBBPe2
9,AGAP08NR77,1_UBBPg3a,1_UBBPg3a


In [ ]:
df_bbu = df_bbu[['NEName','Config_BBU']].reset_index(drop=True)
df_bbu = df_bbu.drop_duplicates()
df_bbu.head(10)

,NEName,Config_BBU
0,AGAP01,1_UBBPd4
1,AGAP02,1_UBBPe2
2,AGAP03,1_UBBPe2
3,AGAP04,1_UBBPe2
4,AGAP05,1_UBBPd4
5,AGAP06,1_UBBPe2
6,AGAP07,1_UBBPe2
7,AGAP08,1_UBBPd4+1_UBBPe2
9,AGAP08NR77,1_UBBPg3a
10,AGAP09,1_UBBPe2


In [ ]:
final_df = final_df.rename(columns={
    'Base Station Name': 'NEName',
    'Sector Equipment Antenna:Subrack No.': 'Subrack No.'
})
final_df.head()

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0


In [ ]:
df_merged = pd.merge(final_df, df_bbu, on='NEName', how='left')
df_merged.head()

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2


In [ ]:
# Ghép df_merged với df_rru theo NEName và Subrack No.
df_final = pd.merge(df_merged, df_rru, on=['NEName', 'Subrack No.'], how='left')
df_final.head()

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU,Board Name,RF Type
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522


In [ ]:
df_final['RF Type Count'] = df_final.groupby(['NEName', 'Frequency band','Subrack No.','RF Type'])['RF Type'].transform('count')
df_final.head(10)

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU,Board Name,RF Type,RF Type Count
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1
5,VLVL59_CoSite,33.0,VLVL59U4CB,1.0,50.0,CELL_BW_N50,4T4R,502.0,180.0,9700.0,123.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1
6,KGRGL3NR77,11.0,KGRGL3M4BA,3.0,1501.0,CELL_BW_N100,4T4R,289.0,516.0,9700.0,0.0,60.0,1_UBBPe2+1_UBBPg2a,MRRU,RRU5901N,1
7,KGRGL3NR77,12.0,KGRGL3M4BB,3.0,1501.0,CELL_BW_N100,4T4R,288.0,510.0,9700.0,1.0,61.0,1_UBBPe2+1_UBBPg2a,MRRU,RRU5901N,1
8,KGRGL3NR77,13.0,KGRGL3M4BC,3.0,1501.0,CELL_BW_N100,4T4R,290.0,702.0,9700.0,2.0,62.0,1_UBBPe2+1_UBBPg2a,MRRU,RRU5901,1
9,VLBT03,33.0,VLBT03C4CC,1.0,50.0,CELL_BW_N50,4T4R,167.0,354.0,9700.0,123.0,62.0,1_UBBPg2+1_UBBPg1a,MRRU,RRU5522,1


In [ ]:
# Tạo cột mới với giá trị là 1 chia cho RF Type Count
import numpy as np
df_final['RF_Type_Count_Inv'] = np.where(
    df_final['RF Type Count'] == 1, "1",
    np.where(df_final['RF Type Count'] == 2, "0.5",
    np.where(df_final['RF Type Count'] == 4, "0.25",
    np.where(df_final['RF Type Count'] == 3, "1/3", "1/{}".format(df_final['RF Type Count'])))))
df_final.head(10)

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU,Board Name,RF Type,RF Type Count,RF_Type_Count_Inv
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1
5,VLVL59_CoSite,33.0,VLVL59U4CB,1.0,50.0,CELL_BW_N50,4T4R,502.0,180.0,9700.0,123.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1
6,KGRGL3NR77,11.0,KGRGL3M4BA,3.0,1501.0,CELL_BW_N100,4T4R,289.0,516.0,9700.0,0.0,60.0,1_UBBPe2+1_UBBPg2a,MRRU,RRU5901N,1,1
7,KGRGL3NR77,12.0,KGRGL3M4BB,3.0,1501.0,CELL_BW_N100,4T4R,288.0,510.0,9700.0,1.0,61.0,1_UBBPe2+1_UBBPg2a,MRRU,RRU5901N,1,1
8,KGRGL3NR77,13.0,KGRGL3M4BC,3.0,1501.0,CELL_BW_N100,4T4R,290.0,702.0,9700.0,2.0,62.0,1_UBBPe2+1_UBBPg2a,MRRU,RRU5901,1,1
9,VLBT03,33.0,VLBT03C4CC,1.0,50.0,CELL_BW_N50,4T4R,167.0,354.0,9700.0,123.0,62.0,1_UBBPg2+1_UBBPg1a,MRRU,RRU5522,1,1


In [ ]:
df_final['Config_RRU'] = df_final['RF_Type_Count_Inv'].astype(str) + '_' + df_final['RF Type'].astype(str)
df_final.head() 

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU,Board Name,RF Type,RF Type Count,RF_Type_Count_Inv,Config_RRU
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,CELL_BW_N100,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,CELL_BW_N75,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,CELL_BW_N100,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,CELL_BW_N50,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,CELL_BW_N50,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522


In [ ]:
BW_map = {'CELL_BW_N100': '20 MHz','CELL_BW_N75': '15 MHz',
          'CELL_BW_N50': '10 MHz','CELL_BW_N25': '5 MHz',}
df_final['Downlink bandwidth'] = df_final['Downlink bandwidth'].map(BW_map)
df_final.head()

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU,Board Name,RF Type,RF Type Count,RF_Type_Count_Inv,Config_RRU
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,20 MHz,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,15 MHz,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,20 MHz,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,10 MHz,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,10 MHz,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522


In [ ]:
df_final['Vendor']= 'Huawei'
df_final.head()

,NEName,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Sector Equipment ID,Subrack No.,Config_BBU,Board Name,RF Type,RF Type Count,RF_Type_Count_Inv,Config_RRU,Vendor
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,20 MHz,4T4R,503.0,726.0,9700.0,0.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,15 MHz,4T4R,186.0,752.0,9700.0,1.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,20 MHz,4T4R,502.0,180.0,9700.0,2.0,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,10 MHz,4T4R,503.0,726.0,9700.0,103.0,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,10 MHz,4T4R,186.0,752.0,9700.0,113.0,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei


In [ ]:
columns_cellop = [
    'Base Station Name','Local cell ID','Local tracking area ID'
]
columns_cnopta = [
    'Base Station Name', 'Local tracking area ID', 'Tracking area code'
]


# Đọc dữ liệu từ các file
file_cellop = 'LST CELLOP.xlsx'
file_cnopta = 'LST CNOPERATORTA.xlsx'


df_cellop = read_excel_columns(file_cellop, columns_cellop)
df_cnopta = read_excel_columns(file_cnopta, columns_cnopta)

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
columns_enodeb = [
    'Base Station Name','eNodeB ID']
file_enodeb = 'LST ENODEBFUNCTION.xlsx'
df_enodeb = read_excel_columns(file_enodeb, columns_enodeb).dropna()
df_enodeb.head()

c:\Users\Chinhnq\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Base Station Name,eNodeB ID
391,VLVL59_CoSite,723454.0
392,KGRGL3NR77,680179.0
393,VLTO28,723253.0
394,DTTH13,662224.0
395,DTCT17,662178.0


In [ ]:
df_cellop.dropna(inplace=True)
df_cellop.head()

,Base Station Name,Local cell ID,Local tracking area ID
391,VLVL59_CoSite,11.0,0.0
392,VLVL59_CoSite,12.0,0.0
393,VLVL59_CoSite,13.0,0.0
394,VLVL59_CoSite,31.0,0.0
395,VLVL59_CoSite,32.0,0.0


In [ ]:
df_cnopta.dropna(inplace=True)
df_cnopta.head()

,Base Station Name,Local tracking area ID,Tracking area code
391,VLVL59_CoSite,0.0,47970.0
392,KGRGL3NR77,0.0,47670.0
393,DTHU19,0.0,47491.0
394,VLTO01,0.0,47960.0
395,VLLH18,0.0,47930.0


In [ ]:
df_cellta = pd.merge(df_cellop,df_cnopta, on=['Base Station Name','Local tracking area ID'],how='left')
df_cellta = pd.merge(df_cellta, df_enodeb, on='Base Station Name', how='left')
df_cellta.head()

,Base Station Name,Local cell ID,Local tracking area ID,Tracking area code,eNodeB ID
0,VLVL59_CoSite,11.0,0.0,47970.0,723454.0
1,VLVL59_CoSite,12.0,0.0,47970.0,723454.0
2,VLVL59_CoSite,13.0,0.0,47970.0,723454.0
3,VLVL59_CoSite,31.0,0.0,47970.0,723454.0
4,VLVL59_CoSite,32.0,0.0,47970.0,723454.0


In [ ]:
df_cellta.drop(columns=['Local tracking area ID'],inplace=True)
df_cellta.head()

,Base Station Name,Local cell ID,Tracking area code,eNodeB ID
0,VLVL59_CoSite,11.0,47970.0,723454.0
1,VLVL59_CoSite,12.0,47970.0,723454.0
2,VLVL59_CoSite,13.0,47970.0,723454.0
3,VLVL59_CoSite,31.0,47970.0,723454.0
4,VLVL59_CoSite,32.0,47970.0,723454.0


In [ ]:
df_final = df_final.rename(columns={'NEName': 'Base Station Name'})
df_cellta = df_cellta.rename(columns={'Local cell ID': 'Local Cell ID'})
df_final = pd.merge(df_final,df_cellta, on=['Base Station Name','Local Cell ID'],how='left')
df_final.head()

,Base Station Name,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),...,Subrack No.,Config_BBU,Board Name,RF Type,RF Type Count,RF_Type_Count_Inv,Config_RRU,Vendor,Tracking area code,eNodeB ID
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,20 MHz,4T4R,503.0,726.0,9700.0,...,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei,47970.0,723454.0
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,15 MHz,4T4R,186.0,752.0,9700.0,...,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei,47970.0,723454.0
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,20 MHz,4T4R,502.0,180.0,9700.0,...,62.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei,47970.0,723454.0
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,10 MHz,4T4R,503.0,726.0,9700.0,...,60.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei,47970.0,723454.0
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,10 MHz,4T4R,186.0,752.0,9700.0,...,61.0,1_UBBPg1a+1_UBBPg2,MRRU,RRU5522,1,1,1_RRU5522,Huawei,47970.0,723454.0


In [ ]:
df_final_huawei = df_final.drop(columns=['Sector Equipment ID','Board Name','RF Type','RF Type Count','RF_Type_Count_Inv'])
df_final_huawei.head()

,Base Station Name,Local Cell ID,Cell Name,Frequency band,Downlink EARFCN,Downlink bandwidth,Cell transmission and reception mode,Physical cell ID,Root sequence index,Cell radius(m),Subrack No.,Config_BBU,Config_RRU,Vendor,Tracking area code,eNodeB ID
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,20 MHz,4T4R,503.0,726.0,9700.0,60.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,15 MHz,4T4R,186.0,752.0,9700.0,61.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,20 MHz,4T4R,502.0,180.0,9700.0,62.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,10 MHz,4T4R,503.0,726.0,9700.0,60.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,10 MHz,4T4R,186.0,752.0,9700.0,61.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0


In [ ]:
rename_columns = {'Frequency band':'Band', 'Downlink bandwidth':'Bandwidth',
                  'Cell transmission and reception mode':'MIMO',
                  'Physical cell ID':'PCI', 'Root sequence index':'RSI','Downlink EARFCN':'EARFCN','Cell radius(m)':'Cell Radius',
                  'Tracking area code':'TAC', 'Subrack No.':'Subrack'}
df_final_huawei = df_final_huawei.rename(columns=rename_columns)
df_final_huawei.head()

,Base Station Name,Local Cell ID,Cell Name,Band,EARFCN,Bandwidth,MIMO,PCI,RSI,Cell Radius,Subrack,Config_BBU,Config_RRU,Vendor,TAC,eNodeB ID
0,VLVL59_CoSite,11.0,VLVL59U4BA,3.0,1501.0,20 MHz,4T4R,503.0,726.0,9700.0,60.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
1,VLVL59_CoSite,12.0,VLVL59U4BP,3.0,1874.0,15 MHz,4T4R,186.0,752.0,9700.0,61.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
2,VLVL59_CoSite,13.0,VLVL59U4BB,3.0,1501.0,20 MHz,4T4R,502.0,180.0,9700.0,62.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
3,VLVL59_CoSite,31.0,VLVL59U4CA,1.0,50.0,10 MHz,4T4R,503.0,726.0,9700.0,60.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0
4,VLVL59_CoSite,32.0,VLVL59U4CG,1.0,50.0,10 MHz,4T4R,186.0,752.0,9700.0,61.0,1_UBBPg1a+1_UBBPg2,1_RRU5522,Huawei,47970.0,723454.0


In [ ]:
df_final_huawei = df_final_huawei[['Base Station Name', 'Local Cell ID', 'Cell Name', 'Band', 'EARFCN',
       'Bandwidth', 'MIMO','eNodeB ID', 'TAC', 'PCI', 'RSI', 'Cell Radius', 'Vendor', 'Subrack', 'Config_BBU', 'Config_RRU']]


In [ ]:
df_final_huawei.to_excel('4G_Huawei_Inventory_Output.xlsx', index=False)